# Emergence of hexagonal grid cells in a path-integrating recurrent network

>**Sorscher, Mel, Ocko, Giocomo & Ganguli (2023), *[A unified theory for the computational and mechanistic origins of grid cells](https://doi.org/10.1016/j.neuron.2022.10.003)*, Neuron 111, 121–137**.

**Abstract**: Grid cells in the medial entorhinal cortex fire on a
hexagonal lattice of positions, and are thought to support path integration — keeping
track of where you are by adding up your own velocity. Hand-designed circuit models can
produce hexagonal lattices, but only because the hexagon was built in by hand. What if the
circuit is *learned* instead? Earlier work trained recurrent networks to path integrate
and got square lattices or irregular blobs, leaving the impression that something extra
was needed. This paper shows the opposite: a vanilla RNN, trained to report its position
through a realistic place-cell readout, develops clean hexagonal grid cells on its own —
provided two biologically ordinary ingredients are present: nonnegative firing rates and
center-surround place-cell targets. A pattern-formation theory then explains *why* the
hexagon, and only the hexagon, survives these constraints.

## Background

**Grid cells.** When a rat forages in an open arena, individual cells in the medial
entorhinal cortex (MEC) fire whenever the animal crosses any vertex of a hexagonal lattice
that tiles the floor (Hafting et al., 2005). The lattice is remarkably regular, it is
shared across environments, and cells come in modules of increasing spacing. Two questions
have driven the field: what circuit mechanism generates such a pattern, and why *this*
pattern — what is the hexagon good for?

**Path integration.** A prominent hypothesis is that MEC keeps a running estimate of
position by integrating velocity over time — dead reckoning, or *path integration*.
Continuous-attractor models realize this by hand-wiring recurrent connectivity with a
center-surround profile so that the network's stable activity states form a hexagonal
lattice. These models work, but the hexagon is an assumption, not a result: the
connectivity is chosen precisely to produce it.

**Learning instead of hand-design.** A parallel line of work trains recurrent networks on
the path-integration task itself: feed the animal's velocity, ask the network to report
position, and inspect what the hidden units become. Cueva & Wei (2018) found square
lattices; Banino et al. (2018, the DeepMind study) found grid-like units, but their fields
were irregular, with grid scores no higher than low-pass-filtered noise. So trained
networks got close, but not to the striking regularity of the brain.

**This paper.** Sorscher and colleagues show that the missing ingredients were modest.
Train a plain single-layer RNN (not an LSTM) with two realistic constraints — hidden-unit
activities are *nonnegative* (a ReLU nonlinearity, as firing rates must be), and the target
place cells have *center-surround* tuning (a difference of Gaussians, as hippocampal place
cells do relative to their surround) — and regular hexagonal grid cells emerge reliably.
They then explain the result with a theory of pattern formation: the position-encoding
objective amplifies a ring of Fourier modes, the shape of the ring is set by the place-cell
tuning, and the nonlinearity selects which combination of modes on the ring wins.
Nonnegativity singles out three wavevectors at 120° to each other — a hexagon. We reproduce
both the trained network and the theory below.

## Setup

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import scipy.ndimage
import torch
import torch.nn.functional as F
from torch import nn

import neuralrnn  # registers all model families (import side effect)
from neuralrnn import AutoConfig, AutoModel, Objective, Trainer, TrainingArguments, register_model
from neuralrnn.models.ctrnn.configuration_ctrnn import CTRNNConfig
from neuralrnn.models.ctrnn.modeling_ctrnn import CTRNNModel

# ---------------------------------------------------------------- paths
NOTEBOOK_DIR = Path.cwd()
MODEL_DIR = NOTEBOOK_DIR / "models" / "grid"
FIG_DIR = NOTEBOOK_DIR / "figs" / "grid"
FIG_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0

# ------------------------------------------------------------- plotting
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 200,
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

print(f"device: {DEVICE}")

## The task: path integration on a random walk

A simulated animal forages in a square box of side $L = 2.2$ m. Its heading $\theta_t$
performs a random walk with Gaussian increments, its speed $s_t$ is drawn from a Rayleigh
distribution, and near the walls it slows down and turns away:

$$
\Delta\theta_t \sim \mathcal{N}(0,\ \sigma_\theta)\,dt,
\qquad \sigma_\theta = 2\times 5.76\ \mathrm{rad/s},
\qquad s_t \sim \mathrm{Rayleigh}(b),\quad b = 0.13\times 2\pi\ \mathrm{m/s},
$$

with time step $dt = 0.02$ s. The network never sees positions — only the instantaneous
velocity

$$
v_t = s_t\, dt\, (\cos\theta_t, \sin\theta_t) \in \mathbb{R}^2 .
$$

**The readout target is a population of place cells.** We place $N_p = 512$ place-cell
centers $c_i$ uniformly at random in the box. The target activity of cell $i$ at position
$x$ is a Gaussian bump, normalized across the population by a softmax, with a broad
surround subtracted — a difference of Gaussians (DoG), or "Mexican hat":

$$
u_i(x) = \mathrm{softmax}_i\!\Big(-\tfrac{\|x - c_i\|^2}{2\sigma^2}\Big),
\qquad
p_i(x) \propto \Big[u_i(x) - \mathrm{softmax}_i\!\Big(-\tfrac{\|x-c_i\|^2}{2\, s_r\, \sigma^2}\Big)\Big]_+^{\ \mathrm{shift}},
$$

with tuning width $\sigma = 0.12$ m and surround ratio $s_r = 2$; the DoG output is shifted
by its minimum and renormalized to sum to one, so the target is a probability distribution
over place cells at every position. This center-surround structure is one of the two
ingredients the paper identifies as essential — keep it in mind for the theory section.

The two modules below implement the walk and the place cells (ported line by line from the
reference implementation).

In [ ]:
"""Random-walk trajectory generation in a square arena.

Faithful PyTorch-free port of ``trajectory_generator.py`` from
Sorscher et al. (2023), "A unified theory of the computational and
mechanistic origins of grid cells" (github.com/ganguli-lab/grid-pattern-formation),
adapted to NeuralRNN's batch-first convention.

A rat-like random walk: at each step the heading turns by a Gaussian
increment and the speed is drawn from a Rayleigh distribution; near the
walls the walk is slowed and steered away. The RNN receives the 2D
velocity of each step and must predict the place-cell code of the
*arrival* position, so the tensors returned here keep the original
alignment:

* ``vel_inputs[b, t]``  = velocity of the move from ``position[t+1]`` to
  ``position[t+2]`` of the underlying walk,
* ``positions[b, t]``   = arrival position (target of the readout),
* ``init_pos[b]``       = position at index 1 (used to seed h0 through
  the place-cell encoder).
"""
from __future__ import annotations

import numpy as np


class TrajectoryGenerator:
    """Random walk in a rectangular box, batch-first output.

    Args:
        sequence_length: Number of steps T per trajectory.
        box_width, box_height: Arena size in meters (centered at origin).
        periodic: If True, wrap positions at the walls (torus) instead of
            wall avoidance.
        border_region: Distance (m) from a wall within which the walk is
            slowed and turned away.
    """

    def __init__(self, sequence_length: int = 20, box_width: float = 2.2,
                 box_height: float = 2.2, periodic: bool = False,
                 border_region: float = 0.03):
        self.sequence_length = sequence_length
        self.box_width = box_width
        self.box_height = box_height
        self.periodic = periodic
        self.border_region = border_region

    # ------------------------------------------------------------------ #
    def _avoid_wall(self, position: np.ndarray, hd: np.ndarray):
        """Distance/angle to nearest wall; turning angle for border cells.

        position: (B, 2), hd: (B,) heading in radians.
        Returns (is_near_wall (B,) bool, turn_angle (B,)).
        """
        x, y = position[:, 0], position[:, 1]
        dists = [self.box_width / 2 - x, self.box_height / 2 - y,
                 self.box_width / 2 + x, self.box_height / 2 + y]
        d_wall = np.min(dists, axis=0)
        theta = (np.arange(4) * np.pi / 2)[np.argmin(dists, axis=0)]
        hd = np.mod(hd, 2 * np.pi)
        a_wall = np.mod(hd - theta + np.pi, 2 * np.pi) - np.pi

        is_near_wall = (d_wall < self.border_region) & (np.abs(a_wall) < np.pi / 2)
        turn_angle = np.zeros_like(hd)
        turn_angle[is_near_wall] = (np.sign(a_wall[is_near_wall])
                                    * (np.pi / 2 - np.abs(a_wall[is_near_wall])))
        return is_near_wall, turn_angle

    # ------------------------------------------------------------------ #
    def generate(self, batch_size: int, rng: np.random.RandomState | None = None):
        """Generate a batch of random-walk trajectories.

        Returns a dict of float32 arrays:
            vel_inputs: (B, T, 2)  cartesian velocity inputs
            positions:  (B, T, 2)  arrival positions (targets)
            init_pos:   (B, 2)     position seeding the initial state
        """
        rng = rng or np.random
        T = self.sequence_length
        dt = 0.02            # time step (s)
        sigma = 5.76 * 2     # angular velocity stdev (rad/s)
        b = 0.13 * 2 * np.pi  # Rayleigh scale of forward speed (m/s)

        position = np.zeros([batch_size, T + 2, 2])
        head_dir = np.zeros([batch_size, T + 2])
        position[:, 0, 0] = rng.uniform(-self.box_width / 2, self.box_width / 2, batch_size)
        position[:, 0, 1] = rng.uniform(-self.box_height / 2, self.box_height / 2, batch_size)
        head_dir[:, 0] = rng.uniform(0, 2 * np.pi, batch_size)
        velocity = np.zeros([batch_size, T + 2])

        random_turn = rng.normal(0.0, sigma, [batch_size, T + 1])
        random_vel = rng.rayleigh(b, [batch_size, T + 1])

        for t in range(T + 1):
            v = random_vel[:, t]
            turn_angle = np.zeros(batch_size)
            if not self.periodic:
                is_near_wall, turn_angle = self._avoid_wall(position[:, t], head_dir[:, t])
                v[is_near_wall] *= 0.25
            turn_angle += dt * random_turn[:, t]

            velocity[:, t] = v * dt
            update = velocity[:, t, None] * np.stack(
                [np.cos(head_dir[:, t]), np.sin(head_dir[:, t])], axis=-1)
            position[:, t + 1] = position[:, t] + update
            head_dir[:, t + 1] = head_dir[:, t] + turn_angle

        if self.periodic:
            position[:, :, 0] = np.mod(position[:, :, 0] + self.box_width / 2,
                                       self.box_width) - self.box_width / 2
            position[:, :, 1] = np.mod(position[:, :, 1] + self.box_height / 2,
                                       self.box_height) - self.box_height / 2

        # Alignment as in the reference: input t is the velocity of the move
        # ending at the target position t (position index t+2).
        ego_v = velocity[:, 1:-1]          # (B, T)
        target_hd = head_dir[:, 1:-1]      # (B, T)
        vel_inputs = np.stack([ego_v * np.cos(target_hd),
                               ego_v * np.sin(target_hd)], axis=-1)

        return {
            "vel_inputs": vel_inputs.astype(np.float32),          # (B, T, 2)
            "positions": position[:, 2:].astype(np.float32),      # (B, T, 2)
            "init_pos": position[:, 1].astype(np.float32),        # (B, 2)
        }

In [ ]:
"""Place-cell target population (ported from ``place_cells.py`` of
Sorscher et al. 2023, github.com/ganguli-lab/grid-pattern-formation).

The supervised target is a population of Np simulated hippocampal place
cells with centers uniformly scattered over the arena. The activity of
the population at position x is

    center   u(x)  = softmax_i( -||x - c_i||^2 / (2 sigma^2) )
    surround s(x)  = softmax_i( -||x - c_i||^2 / (2 * surround_scale * sigma^2) )
    DoG      p(x)  = (u - s); shifted by its minimum and renormalized to sum 1

i.e. a difference-of-gaussians (center-surround, "Mexican hat") tuning
curve, normalized into a probability distribution so the network can be
trained with cross-entropy. The surround is what the theory links to the
emergence of hexagonal (rather than stripe-like) patterns.
"""
from __future__ import annotations

import numpy as np
import torch


class PlaceCells:
    """Softmax-Gaussian (+ optional DoG surround) place-cell population.

    Args:
        n_cells: Number of place cells Np.
        box_width, box_height: Arena size in meters (centered at origin).
        sigma: Width of the center tuning curve (m).
        surround_scale: Ratio of surround to center variance (DoG only).
        dog: Use difference-of-gaussians tuning (paper default True).
        periodic: Torus boundary conditions for distances.
        device: Torch device holding the centers.
        seed: Seed for the (fixed) random center layout.
    """

    def __init__(self, n_cells: int = 512, box_width: float = 2.2,
                 box_height: float = 2.2, sigma: float = 0.12,
                 surround_scale: float = 2.0, dog: bool = True,
                 periodic: bool = False, device: str = "cpu", seed: int = 0):
        self.n_cells = n_cells
        self.sigma = sigma
        self.surround_scale = surround_scale
        self.box_width = box_width
        self.box_height = box_height
        self.dog = dog
        self.periodic = periodic
        self.device = device

        rng = np.random.RandomState(seed)
        usx = rng.uniform(-box_width / 2, box_width / 2, (n_cells,))
        usy = rng.uniform(-box_height / 2, box_height / 2, (n_cells,))
        self.centers = torch.tensor(np.vstack([usx, usy]).T,
                                    dtype=torch.float32, device=device)

    # ------------------------------------------------------------------ #
    def get_activation(self, pos: torch.Tensor) -> torch.Tensor:
        """Place-cell population activity at given positions.

        pos: (..., 2) torch tensor. Returns (..., Np), rows sum to 1.
        """
        pos = pos.to(self.device).float()
        d = torch.abs(pos[..., None, :] - self.centers)  # (..., Np, 2)
        if self.periodic:
            dx = torch.minimum(d[..., 0], self.box_width - d[..., 0])
            dy = torch.minimum(d[..., 1], self.box_height - d[..., 1])
            d = torch.stack([dx, dy], dim=-1)
        norm2 = (d ** 2).sum(-1)                          # (..., Np)

        outputs = torch.softmax(-norm2 / (2 * self.sigma ** 2), dim=-1)
        if self.dog:
            outputs = outputs - torch.softmax(
                -norm2 / (2 * self.surround_scale * self.sigma ** 2), dim=-1)
            min_output, _ = outputs.min(-1, keepdim=True)
            outputs = outputs + torch.abs(min_output)
            outputs = outputs / outputs.sum(-1, keepdim=True)
        return outputs

    # ------------------------------------------------------------------ #
    def decode(self, activation: torch.Tensor, k: int = 3) -> torch.Tensor:
        """Decode position as the mean center of the k most active cells.

        activation: (..., Np). Returns (..., 2).
        """
        _, idxs = torch.topk(activation, k=k, dim=-1)
        return self.centers[idxs].mean(-2)

    # ------------------------------------------------------------------ #
    def covariance(self, res: int = 30) -> np.ndarray:
        """Spatial covariance kernel of the place-cell population.

        C(dx, dy) = average over positions x of <p(x), p(x + (dx,dy))>,
        computed by brute-force circulant averaging on a res x res grid
        (paper Fig. on pattern formation uses res=55). Returns (res, res).
        """
        xs = np.linspace(-self.box_width / 2, self.box_width / 2, res)
        ys = np.linspace(-self.box_height / 2, self.box_height / 2, res)
        pos = np.array(np.meshgrid(xs, ys)).T             # (res, res, 2)
        pos_t = torch.tensor(pos, dtype=torch.float32, device=self.device)
        pc = self.get_activation(pos_t).reshape(-1, self.n_cells).cpu().numpy()

        C = pc @ pc.T
        Csquare = C.reshape(res, res, res, res)
        Cmean = np.zeros([res, res])
        for i in range(res):
            for j in range(res):
                Cmean += np.roll(np.roll(Csquare[i, j], -i, axis=0), -j, axis=1)
        Cmean = np.roll(np.roll(Cmean, res // 2, axis=0), res // 2, axis=1)
        return Cmean

Let us look at the data the network will live on: one long trajectory, and the
tuning curves of a few place cells.

In [ ]:
# ------------------------------------------------- build the task world
traj_gen = TrajectoryGenerator(sequence_length=20, box_width=2.2, box_height=2.2)
place_cells = PlaceCells(n_cells=512, box_width=2.2, box_height=2.2,
                         sigma=0.12, surround_scale=2.0, dog=True, device=DEVICE)

# a long trajectory purely for illustration (T = 3000 steps = 60 s)
demo_gen = TrajectoryGenerator(sequence_length=3000, box_width=2.2, box_height=2.2)
demo = demo_gen.generate(batch_size=1, rng=np.random.RandomState(3))
demo_pos = np.concatenate([demo["init_pos"], demo["positions"][0]], axis=0)  # (T+1, 2)

# place-cell tuning curves evaluated on a fine grid covering the box
res_fine = 100
gx, gy = np.meshgrid(np.linspace(-1.1, 1.1, res_fine), np.linspace(-1.1, 1.1, res_fine))
grid_pos = torch.tensor(np.stack([gx.ravel(), gy.ravel()], -1), dtype=torch.float32)
tuning = place_cells.get_activation(grid_pos.to(DEVICE)).cpu().numpy()      # (res^2, Np)
tuning = tuning.T.reshape(-1, res_fine, res_fine)                            # (Np, res, res)
show_cells = [0, 40, 120, 200, 300, 450]  # arbitrary place cells to display

# 1-D profile of the DoG kernel: tuning of one cell along the line through its center
cell_center = place_cells.centers[0].cpu().numpy()
rs = np.linspace(-0.6, 0.6, 200)
line_pos = torch.tensor(np.stack([cell_center[0] + rs,
                                  np.full_like(rs, cell_center[1])], -1))
line_act = place_cells.get_activation(line_pos.to(DEVICE))[:, 0].cpu().numpy()

fig = plt.figure(figsize=(10.5, 3.4))
gs = plt.GridSpec(2, 6, width_ratios=[1.9, 1, 1, 1, 1.2, 1.3], hspace=0.45, wspace=0.35)

# panel a: the random-walk path
ax_traj = fig.add_subplot(gs[:, 0])
ax_traj.plot(demo_pos[:, 0], demo_pos[:, 1], lw=0.35, color="0.3")
ax_traj.scatter(*demo_pos[0], c="tab:green", s=18, zorder=5, label="start")
ax_traj.scatter(*demo_pos[-1], c="tab:red", s=18, zorder=5, label="end")
ax_traj.set(xlim=(-1.1, 1.1), ylim=(-1.1, 1.1), xlabel="x (m)", ylabel="y (m)",
            title="simulated foraging path (60 s)", aspect="equal")
ax_traj.legend(loc="upper right", fontsize=6.5)

# panel b: six example DoG place-cell tuning curves
for k, cell_i in enumerate(show_cells):
    ax = fig.add_subplot(gs[k // 3, 1 + k % 3])
    ax.imshow(tuning[cell_i], cmap="jet", origin="lower", extent=[-1.1, 1.1, -1.1, 1.1])
    ax.set_title(f"place cell {cell_i}", fontsize=7)
    ax.set_xticks([]); ax.set_yticks([])

# panel c: 1-D slice through one cell's DoG profile
ax_prof = fig.add_subplot(gs[:, 4])
ax_prof.plot(rs, line_act, color="k", lw=1)
ax_prof.set(xlabel="dist. from center (m)", ylabel="activity", xlim=(-0.6, 0.6),
            title="DoG tuning profile\n(1-D slice through center)")
ax_prof.tick_params(labelsize=6)

# panel d: speed distribution of the walk
ax_spd = fig.add_subplot(gs[:, 5])
speeds = np.linalg.norm(demo["vel_inputs"][0], axis=-1) / 0.02   # m/s
ax_spd.hist(speeds, bins=40, color="0.4")
ax_spd.set(xlabel="speed (m/s)", ylabel="count",
           title="Rayleigh speed\ndistribution")
ax_spd.tick_params(labelsize=6)

fig.suptitle("The task world: random foraging in a 2.2 m box, read out by "
             "center-surround place cells", fontsize=10, y=1.02)
fig.savefig(FIG_DIR / "task.png", bbox_inches="tight")
plt.show()

## The network: a vanilla RNN with a place-cell readout

The model is deliberately plain — the point of the paper is that nothing exotic is needed.
A single-layer recurrent network with $N_g = 4096$ hidden units $g_t \in \mathbb{R}^{N_g}$
receives the velocity $v_t$, and starts each trajectory from an encoding of the initial
position through the place-cell population:

$$
g_0 = W_{\mathrm{enc}}\, p(x_0), \qquad
g_t = \mathrm{ReLU}\!\big(W_{\mathrm{rec}}\, g_{t-1} + W_{\mathrm{in}}\, v_t\big), \qquad
\hat{y}_t = \mathrm{softmax}\!\big(W_{\mathrm{out}}\, g_t\big),
$$

where all weight matrices are bias-free. The readout $\hat y_t$ is a distribution over the
$N_p$ place cells, trained to match the DoG target $y_t = p(x_t)$ with a soft-target
cross-entropy, plus weight decay on the recurrent weights only:

$$
\mathcal{L} = -\sum_{t} \sum_{i=1}^{N_p} y_{t,i} \log \hat{y}_{t,i}
\;+\; \lambda \lVert W_{\mathrm{rec}} \rVert_F^2,
\qquad \lambda = 10^{-4},
$$

optimized with Adam (learning rate $10^{-4}$, no gradient clipping), on freshly generated
trajectories at every step (20 steps per trajectory, 200 trajectories per batch). The ReLU
is the first essential ingredient: it makes hidden activations *nonnegative*, like firing
rates. The weight decay is the regularizer the paper sweeps in its Fig. 1E — a small amount
is what allows the network to keep path integrating far outside the walls of the training
arena, and it also sharpens the grids.

**Framework mapping.** With the Euler fraction $\alpha = 1$ (`dt=None`), the framework's
`CTRNNModel` is *exactly* the discrete-time recurrence above. The only genuinely new piece
is the encoder $W_{\mathrm{enc}}$ that seeds $g_0$ from the initial place-cell activity; we
add it as a small subclass and pass its output through the framework's
`forward(inputs, initial_state=...)` hook. Training uses the framework `Trainer` with a
small custom `Objective` implementing the loss above (the stock classification objective
expects hard labels; here targets are full distributions).

In [ ]:
"""Path-integration RNN as a NeuralRNN model family (Sorscher et al. 2023).

The paper's network is a discrete-time vanilla RNN

    g_t = f(W_rec g_{t-1} + W_in v_t),          f = ReLU (or tanh)
    h_0 = W_enc p(x_0),                          initial position seed
    yhat_t = softmax(W_out g_t)                  predicted place-cell code

trained with the soft-target cross-entropy  -sum_i y_i log yhat_i  plus an
L2 penalty on the recurrent weights only.

Mapping onto the NeuralRNN framework:

* ``GridRNN`` subclasses the stock :class:`CTRNNModel` with ``dt=None``
  (alpha = 1), which is exactly the discrete-time equation above. The only
  addition is the ``encoder`` Linear(Np -> Ng) that turns the initial
  place-cell activity into the initial hidden state; it is passed through
  the framework's ``forward(inputs, initial_state=...)`` hook. Following
  the paper, all layers are bias-free (biases zeroed and frozen).
* ``GridCellObjective`` implements the loss. Soft targets rule out the
  stock classification objective (hard labels), so this is a small
  ``Objective`` subclass — the framework's designated extension point.
* ``PathIntegrationDataset`` provides ``sample_batch()`` (the only method
  the framework ``Trainer`` needs) by drawing fresh random-walk
  trajectories on every call.
"""
from __future__ import annotations

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

from neuralrnn import Objective, register_model
from neuralrnn.models.ctrnn.configuration_ctrnn import CTRNNConfig
from neuralrnn.models.ctrnn.modeling_ctrnn import CTRNNModel



class GridRNNConfig(CTRNNConfig):
    """CTRNN config reused verbatim; only the registry name differs."""

    model_type = "grid_rnn"


@register_model("grid_rnn")
class GridRNN(CTRNNModel):
    """CTRNN (alpha = 1, discrete time) + place-cell encoder for h0.

    Adds ``encoder: Linear(Np -> Ng, bias=False)`` mapping the place-cell
    activity at the starting position to the initial hidden state, exactly
    as in the reference model. All biases are zeroed and frozen (the paper
    uses bias-free layers throughout).
    """

    config_class = GridRNNConfig

    def __init__(self, config: GridRNNConfig):
        super().__init__(config)
        self.encoder = nn.Linear(config.output_dim, config.latent_dim, bias=False)
        for layer in (self.input2h, self.h2h, self.readout_layer):
            nn.init.zeros_(layer.bias)
            layer.bias.requires_grad_(False)

    def forward(self, inputs: torch.Tensor, init_pc: torch.Tensor | None = None,
                **kwargs):
        """inputs: (B, T, 2) velocities; init_pc: (B, Np) place activity at t=0."""
        initial_state = self.encoder(init_pc) if init_pc is not None else None
        return super().forward(inputs, initial_state=initial_state, **kwargs)


class GridCellObjective(Objective):
    """Soft-target cross-entropy + L2 on the recurrent weights only.

    loss = -(y * log softmax(logits)).sum(-1).mean() + weight_decay * ||W_rec||^2

    Also reports the position decoding error (top-3 place-cell readout, cm).
    """

    def __init__(self, place_cells: PlaceCells, weight_decay: float = 1e-4):
        self.place_cells = place_cells
        self.weight_decay = weight_decay

    def compute_loss(self, model: GridRNN, batch: dict[str, torch.Tensor]):
        out = model(batch["inputs"], init_pc=batch["init_pc"])
        logits = out.outputs                                  # (B, T, Np)
        y = batch["targets"]                                  # (B, T, Np)

        ce = -(y * F.log_softmax(logits, dim=-1)).sum(-1).mean()
        l2 = (model.h2h.weight ** 2).sum()
        loss = ce + self.weight_decay * l2

        with torch.no_grad():
            pred_pos = self.place_cells.decode(logits)        # (B, T, 2)
            err = torch.sqrt(((batch["positions"] - pred_pos) ** 2).sum(-1)).mean()
        return loss, {"loss": float(loss), "ce": float(ce),
                      "decode_err_cm": 100 * float(err)}


class PathIntegrationDataset:
    """On-the-fly dataset: every batch is a fresh set of random walks.

    Only ``sample_batch()`` is required by the framework Trainer.
    """

    def __init__(self, trajectory_generator: TrajectoryGenerator,
                 place_cells: PlaceCells, batch_size: int = 200,
                 rng: np.random.RandomState | None = None):
        self.trajectory_generator = trajectory_generator
        self.place_cells = place_cells
        self.batch_size = batch_size
        self.rng = rng or np.random.RandomState(0)

    def sample_batch(self) -> dict[str, torch.Tensor]:
        traj = self.trajectory_generator.generate(self.batch_size, rng=self.rng)
        pos = torch.tensor(traj["positions"])                 # (B, T, 2)
        with torch.no_grad():
            targets = self.place_cells.get_activation(pos)    # (B, T, Np)
            init_pc = self.place_cells.get_activation(
                torch.tensor(traj["init_pos"]))               # (B, Np)
        return {
            "inputs": torch.tensor(traj["vel_inputs"]),       # (B, T, 2)
            "targets": targets.cpu(),
            "init_pc": init_pc.cpu(),
            "positions": pos,
        }

In [ ]:
# ------------------------------------------------- build model & dataset
N_GRID, N_PLACE = 4096, 512          # hidden units / place cells (paper defaults)

config = AutoConfig.for_model(
    "grid_rnn", input_dim=2, latent_dim=N_GRID, output_dim=N_PLACE,
    activation="relu")               # dt=None -> alpha=1: discrete-time vanilla RNN
model = AutoModel.from_config(config)

dataset = PathIntegrationDataset(traj_gen, place_cells, batch_size=200,
                                 rng=np.random.RandomState(SEED))
objective = GridCellObjective(place_cells, weight_decay=1e-4)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"GridRNN: {N_GRID} hidden units, {N_PLACE} place cells, "
      f"{n_params/1e6:.1f}M trainable parameters")
print(f"alpha = {config.alpha} (discrete-time limit), activation = {config.activation}")

batch = dataset.sample_batch()
print("batch shapes:", {k: tuple(v.shape) for k, v in batch.items()})

## Training

We train for 100,000 steps — the paper's 100 epochs of 1000 batches — with fresh random
trajectories every step, and every 2000 steps we measure the mean grid score of a subset of
units (defined precisely in the next section) so we can watch the lattice crystallize. The
loss has a long flat plateau at the chance level $\log N_p$ while the readout slowly breaks
symmetry; the original implementation shows the same plateau.

Training is **load-first**: if `models/grid/` already holds a completed run, the cell below
reloads it in seconds. Otherwise it runs the full training (about three hours on a GPU).

In [ ]:
"""Training driver with the load-first convention.

``train_or_load`` returns the saved model unchanged when ``out_dir``
already holds a completed run (``config.json`` + ``run_meta.json``, the
latter written last as a completion flag); otherwise it trains with the
framework Trainer and checkpoints. Training curves (``history.json``) and
periodic grid-score evaluations (``eval_history.json``) are stored next to
the weights so the notebook can plot learning without retraining.
"""
from __future__ import annotations

import json
import time
from pathlib import Path

import numpy as np
import torch

from neuralrnn import AutoConfig, AutoModel, Trainer, TrainingArguments


# Paper defaults (main.py of the reference implementation)
PAPER = dict(
    box_width=2.2, box_height=2.2,       # arena (m)
    sigma=0.12,                          # place-cell tuning width (m)
    surround_scale=2.0,                  # DoG surround ratio
    dog=True,                            # difference-of-gaussians targets
    n_grid=4096,                         # hidden units Ng
    n_place=512,                         # place cells Np
    seq_len=20,                          # steps per trajectory
    batch_size=200,
    lr=1e-4,
    weight_decay=1e-4,                   # L2 on W_rec only (in the objective)
    activation="relu",
)


def build_world(cfg: dict, device: str, seed: int = 0):
    """Place cells + trajectory generator + dataset from a config dict."""
    place_cells = PlaceCells(n_cells=cfg["n_place"], box_width=cfg["box_width"],
                             box_height=cfg["box_height"], sigma=cfg["sigma"],
                             surround_scale=cfg["surround_scale"], dog=cfg["dog"],
                             device=device)
    traj_gen = TrajectoryGenerator(sequence_length=cfg["seq_len"],
                                   box_width=cfg["box_width"],
                                   box_height=cfg["box_height"])
    dataset = PathIntegrationDataset(traj_gen, place_cells,
                                     batch_size=cfg["batch_size"],
                                     rng=np.random.RandomState(seed))
    return place_cells, traj_gen, dataset


def make_grid_eval_fn(dataset, device: str, box_width: float,
                      n_batches: int = 5, n_units: int = 128, res: int = 20):
    """eval_fn for the Trainer: mean grid score over a subset of units."""
    def eval_fn(model):
        ratemaps, _ = compute_ratemaps(model, dataset, res=res,
                                       n_batches=n_batches,
                                       unit_idxs=np.arange(n_units))
        scores = score_ratemaps(ratemaps, box_width)
        model.train()
        return {"mean_grid_score_60": float(np.mean(scores["score_60"])),
                "frac_grid_cells": float(np.mean(scores["score_60"] > 0.5))}
    return eval_fn


def train_or_load(out_dir, device: str = "cuda", max_steps: int = 50_000,
                  seed: int = 0, eval_every: int = 2_000, log_every: int = 200,
                  **overrides):
    """Load a completed run from ``out_dir`` or train it with the Trainer.

    Returns (model, meta) where meta holds the config, wall time, and the
    training/eval histories.
    """
    out_dir = Path(out_dir)
    meta_path = out_dir / "run_meta.json"
    if meta_path.exists() and (out_dir / "config.json").exists():
        model = AutoModel.from_pretrained(str(out_dir))
        model.to(device)
        meta = json.loads(meta_path.read_text())
        meta["loaded_from_checkpoint"] = True
        return model, meta

    cfg = {**PAPER, **overrides}
    place_cells, _, dataset = build_world(cfg, device, seed)

    config = AutoConfig.for_model(
        "grid_rnn", input_dim=2, latent_dim=cfg["n_grid"],
        output_dim=cfg["n_place"], activation=cfg["activation"])  # dt=None -> alpha=1
    model = AutoModel.from_config(config)

    objective = GridCellObjective(place_cells, weight_decay=cfg["weight_decay"])
    args = TrainingArguments(
        learning_rate=cfg["lr"], max_steps=max_steps, batch_size=cfg["batch_size"],
        optimizer="adam", weight_decay=0.0,        # L2 on W_rec lives in the objective
        grad_clip_norm=None,                       # no clipping, as in the paper
        log_every=log_every, eval_every=eval_every,
        output_dir=str(out_dir), device=device, seed=seed)
    eval_fn = make_grid_eval_fn(dataset, device, cfg["box_width"])
    trainer = Trainer(model, dataset, objective, args, eval_fn=eval_fn)

    t0 = time.time()
    trainer.train()
    wall_time = time.time() - t0

    out_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(out_dir))
    (out_dir / "history.json").write_text(json.dumps(trainer.history))
    (out_dir / "eval_history.json").write_text(json.dumps(trainer._eval_history))
    meta = {"config": cfg, "max_steps": max_steps, "seed": seed,
            "wall_time_s": wall_time, "loaded_from_checkpoint": False}
    meta_path.write_text(json.dumps(meta, indent=2))   # completion flag, written last
    return model, meta

In [ ]:
model, meta = train_or_load(MODEL_DIR, device=DEVICE, max_steps=100_000,
                            seed=SEED, eval_every=2_000, log_every=500)
if meta["loaded_from_checkpoint"]:
    print(f"loaded checkpoint trained for {meta['max_steps']} steps "
          f"(wall time {meta['wall_time_s']/3600:.1f} h)")
else:
    print(f"trained {meta['max_steps']} steps in {meta['wall_time_s']/3600:.1f} h")

history = json.loads((MODEL_DIR / "history.json").read_text())
eval_history = json.loads((MODEL_DIR / "eval_history.json").read_text())

In [ ]:
steps = [h["step"] for h in history]
ce = [h["ce"] for h in history]
err = [h["decode_err_cm"] for h in history]
ev_steps = [e["step"] for e in eval_history]
ev_grid = [e["mean_grid_score_60"] for e in eval_history]
ev_frac = [e["frac_grid_cells"] for e in eval_history]

fig, axes = plt.subplots(1, 3, figsize=(9.5, 2.6))
ax = axes[0]
ax.plot(steps, ce, lw=1, color="k")
ax.axhline(np.log(N_PLACE), ls="--", lw=0.7, color="0.6", label=f"chance level ln({N_PLACE})")
ax.set(xlabel="training step", ylabel="cross-entropy", title="readout loss")
ax.legend(fontsize=7)

ax = axes[1]
ax.plot(steps, err, lw=1, color="tab:red")
ax.set(xlabel="training step", ylabel="decoding error (cm)",
       title="position decoding from the readout")

ax = axes[2]
ax.plot(ev_steps, ev_grid, lw=1.2, color="tab:blue", marker="o", ms=3,
        label="mean grid score (128 units)")
ax2 = ax.twinx()
ax2.plot(ev_steps, ev_frac, lw=1.2, color="tab:green", marker="s", ms=3,
         label="fraction grid cells (>0.5)")
ax2.spines["right"].set_visible(True)
ax.set(xlabel="training step", ylabel="mean grid score", title="grid cells crystallize")
ax2.set_ylabel("fraction of grid cells", color="tab:green")
ax.legend(loc="center left", fontsize=7)

fig.suptitle("Learning curves: the position code sharpens, then the hexagon appears",
             fontsize=10, y=1.04)
fig.savefig(FIG_DIR / "training.png", bbox_inches="tight")
plt.show()

## Hexagonal grid cells emerge

Now for the central result. We let the trained network path integrate fresh random walks,
bin each hidden unit's activity over the arena into a **ratemap**, and quantify six-fold
symmetry with the standard **grid score** (Langston et al., 2010; the implementation is the
DeepMind/ganguli-lab one): take the spatial autocorrelogram (SAC) of the ratemap, rotate it
by several angles, and correlate the rotated SAC with the original inside the best annulus:

$$
\mathrm{score}_{60} = \min\big(\rho_{60},\ \rho_{120}\big) - \max\big(\rho_{30},\ \rho_{90},\ \rho_{150}\big),
\qquad
\mathrm{score}_{90} = \rho_{90} - \tfrac{\rho_{45} + \rho_{135}}{2},
$$

where $\rho_{\theta}$ is the correlation under rotation by $\theta$ degrees. A hexagonal
lattice looks the same after a 60° rotation but not after 30°, so hexagons score high on
$\mathrm{score}_{60}$ and squares on $\mathrm{score}_{90}$.

In [ ]:
"""Grid scores via the spatial autocorrelogram (SAC).

Ported from ``scores.py`` of Sorscher et al. 2023 (which itself is the
Apache-2.0 DeepMind grid-cells implementation, Copyright 2018 Google LLC).

A ratemap is correlated with rotated copies of its own spatial
autocorrelogram: hexagonal patterns correlate at 60/120 deg but not at
30/90/150 deg, square patterns at 90 deg. Only the numerical core is kept
(no plotting).
"""
from __future__ import annotations

import math

import numpy as np
import scipy.ndimage
import scipy.signal
import scipy.stats


def circle_mask(size, radius, in_val=1.0, out_val=0.0):
    """Binary disc mask of given radius (in samples) on a size x size array."""
    sz = [math.floor(size[0] / 2), math.floor(size[1] / 2)]
    x = np.linspace(-sz[0], sz[1], size[1])
    x = np.expand_dims(x, 0).repeat(size[0], 0)
    y = np.expand_dims(np.linspace(-sz[0], sz[1], size[1]), 1)
    y = y.repeat(size[1], 1)
    z = np.sqrt(x ** 2 + y ** 2)
    return np.where(z <= radius, in_val, out_val)


class GridScorer:
    """Grid scores (60 and 90 deg) for ratemaps of ``nbins`` per dimension.

    Args:
        nbins: Ratemap resolution (bins per dimension).
        coords_range: ((xmin, xmax), (ymin, ymax)) of the arena.
        mask_parameters: iterable of (min_radius, max_radius) ring masks,
            radii as a fraction of the arena width.
        min_max: If True use the stricter min-max definition of score_60.
    """

    def __init__(self, nbins: int, coords_range, mask_parameters, min_max: bool = False):
        self._nbins = nbins
        self._min_max = min_max
        self._coords_range = coords_range
        self._corr_angles = [30, 45, 60, 90, 120, 135, 150]
        self._masks = [(self._get_ring_mask(lo, hi), (lo, hi))
                       for lo, hi in mask_parameters]

    def calculate_ratemap(self, xs, ys, activations, statistic="mean"):
        """Binned ratemap from samples (xs, ys) and unit activations."""
        return scipy.stats.binned_statistic_2d(
            xs, ys, activations, bins=self._nbins, statistic=statistic,
            range=self._coords_range)[0]

    def _get_ring_mask(self, mask_min, mask_max):
        n_points = [self._nbins * 2 - 1, self._nbins * 2 - 1]
        return (circle_mask(n_points, mask_max * self._nbins)
                * (1 - circle_mask(n_points, mask_min * self._nbins)))

    def grid_score_60(self, corr):
        if self._min_max:
            return np.minimum(corr[60], corr[120]) - np.maximum(
                corr[30], np.maximum(corr[90], corr[150]))
        return (corr[60] + corr[120]) / 2 - (corr[30] + corr[90] + corr[150]) / 3

    def grid_score_90(self, corr):
        return corr[90] - (corr[45] + corr[135]) / 2

    def calculate_sac(self, seq1):
        """Spatial autocorrelogram (Pearson coefficient field)."""
        seq2 = seq1

        def filter2(b, x):
            stencil = np.rot90(b, 2)
            return scipy.signal.convolve2d(x, stencil, mode="full")

        seq1 = np.nan_to_num(seq1)
        seq2 = np.nan_to_num(seq2)
        ones = np.ones(seq1.shape)

        seq1_sq = np.square(seq1)
        seq2_sq = np.square(seq2)

        seq1_x_seq2 = filter2(seq1, seq2)
        sum_seq1 = filter2(seq1, ones)
        sum_seq2 = filter2(ones, seq2)
        sum_seq1_sq = filter2(seq1_sq, ones)
        sum_seq2_sq = filter2(ones, seq2_sq)
        n_bins = filter2(ones, ones)
        n_bins_sq = np.square(n_bins)

        std_seq1 = np.power(np.subtract(
            np.divide(sum_seq1_sq, n_bins),
            np.divide(np.square(sum_seq1), n_bins_sq)), 0.5)
        std_seq2 = np.power(np.subtract(
            np.divide(sum_seq2_sq, n_bins),
            np.divide(np.square(sum_seq2), n_bins_sq)), 0.5)
        covar = np.subtract(
            np.divide(seq1_x_seq2, n_bins),
            np.divide(np.multiply(sum_seq1, sum_seq2), n_bins_sq))
        x_coef = np.divide(covar, np.multiply(std_seq1, std_seq2))
        return np.nan_to_num(np.real(x_coef))

    def rotated_sacs(self, sac, angles):
        return [scipy.ndimage.rotate(sac, angle, reshape=False) for angle in angles]

    def get_grid_scores_for_mask(self, sac, rotated_sacs, mask):
        """Pearson correlations inside one ring mask at all rotation angles."""
        masked_sac = sac * mask
        ring_area = np.sum(mask)
        masked_sac_mean = np.sum(masked_sac) / ring_area
        masked_sac_centered = (masked_sac - masked_sac_mean) * mask
        variance = np.sum(masked_sac_centered ** 2) / ring_area + 1e-5
        corrs = {}
        for angle, rotated_sac in zip(self._corr_angles, rotated_sacs):
            masked_rotated_sac = (rotated_sac - masked_sac_mean) * mask
            cross_prod = np.sum(masked_sac_centered * masked_rotated_sac) / ring_area
            corrs[angle] = cross_prod / variance
        return self.grid_score_60(corrs), self.grid_score_90(corrs), variance

    def get_scores(self, rate_map):
        """Returns (score_60, score_90, mask_60, mask_90, sac, best_mask_idx)."""
        sac = self.calculate_sac(rate_map)
        rotated = self.rotated_sacs(sac, self._corr_angles)
        scores = [self.get_grid_scores_for_mask(sac, rotated, mask)
                  for mask, _ in self._masks]
        scores_60, scores_90, _ = map(np.asarray, zip(*scores))
        i60, i90 = np.argmax(scores_60), np.argmax(scores_90)
        return (scores_60[i60], scores_90[i90], self._masks[i60][1],
                self._masks[i90][1], sac, i60)


def default_scorer(res: int, box_width: float) -> GridScorer:
    """The standard scorer: ring masks from 0.2 to 1.0 of the arena width."""
    half = box_width / 2
    coords_range = ((-half, half), (-half, half))
    starts = [0.2] * 10
    ends = np.linspace(0.4, 1.0, num=10)
    return GridScorer(res, coords_range, list(zip(starts, ends.tolist())))

In [ ]:
"""Analysis utilities: ratemaps, grid-score sweeps, pattern formation.

Ratemap computation follows ``visualize.compute_ratemaps`` of Sorscher et
al. 2023 (histogram of hidden-unit activity over spatial bins). The
pattern-formation simulations follow ``pattern_formation.ipynb``: gradient
dynamics on the position-encoding objective,

    dG/dt = -G + conv(G, Ctilde),   G <- phi(G)

where C is the place-cell covariance kernel and phi the nonlinearity
(tanh preserves the g -> -g symmetry and yields stripes; ReLU breaks it
and yields hexagonal blobs).
"""
from __future__ import annotations

import numpy as np
import torch



@torch.no_grad()
def compute_ratemaps(model, dataset, res: int = 20, n_batches: int = 50,
                     unit_idxs: np.ndarray | None = None):
    """Spatial ratemaps of the hidden units from fresh random walks.

    Args:
        model: Trained GridRNN.
        dataset: PathIntegrationDataset used to draw test batches.
        res: Ratemap resolution (bins per dimension).
        n_batches: Number of batches to average over.
        unit_idxs: Subset of hidden units (default all).

    Returns:
        ratemaps: (n_units, res, res) mean activation per spatial bin.
        occupancy: (res, res) sample counts per bin.
    """
    model.eval()
    device = next(model.parameters()).device
    gen = dataset.trajectory_generator
    pc = dataset.place_cells

    first = dataset.sample_batch()
    n_units_all = model.config.latent_dim
    idxs = np.arange(n_units_all) if unit_idxs is None else np.asarray(unit_idxs)
    n_units = len(idxs)

    activations = np.zeros([n_units, res, res])
    occupancy = np.zeros([res, res])

    def accumulate(batch):
        inputs = batch["inputs"].to(device)
        init_pc = batch["init_pc"].to(device)
        states = model(inputs, init_pc=init_pc).states          # (B, T, Ng)
        g = states[:, :, idxs].reshape(-1, n_units).cpu().numpy()
        pos = batch["positions"].reshape(-1, 2).numpy()
        x_bin = (pos[:, 0] + gen.box_width / 2) / gen.box_width * res
        y_bin = (pos[:, 1] + gen.box_height / 2) / gen.box_height * res
        valid = (x_bin >= 0) & (x_bin < res) & (y_bin >= 0) & (y_bin < res)
        xi, yi = x_bin[valid].astype(int), y_bin[valid].astype(int)
        flat = xi * res + yi
        np.add.at(occupancy.reshape(-1), flat, 1)
        # activations[:, x, y] += g for each sample (vectorized scatter-add)
        rows = np.arange(n_units)[:, None]
        np.add.at(activations.reshape(n_units, -1),
                  (rows, flat[None, :]), g[valid].T)

    accumulate(first)
    for _ in range(n_batches - 1):
        accumulate(dataset.sample_batch())

    filled = occupancy > 0
    activations[:, filled] /= occupancy[filled]
    return activations, occupancy


def score_ratemaps(ratemaps: np.ndarray, box_width: float):
    """Grid scores for a stack of ratemaps (n_units, res, res).

    Returns dict with score_60, score_90 (each (n_units,)) and the list of
    SACs. Bins with zero occupancy are set to zero before scoring.
    """
    res = ratemaps.shape[-1]
    scorer = default_scorer(res, box_width)
    s60, s90, sacs = [], [], []
    for rm in ratemaps:
        rm = np.nan_to_num(rm)
        score_60, score_90, _, _, sac, _ = scorer.get_scores(rm)
        s60.append(score_60)
        s90.append(score_90)
        sacs.append(sac)
    return {"score_60": np.asarray(s60), "score_90": np.asarray(s90),
            "sacs": sacs, "scorer": scorer}


# --------------------------------------------------------------------- #
# Pattern formation (numerical simulation of the theory)
# --------------------------------------------------------------------- #
def covariance_ft(C: np.ndarray) -> np.ndarray:
    """Fourier transform of the covariance kernel with the DC mode removed."""
    Ctilde = np.fft.fft2(C)
    Ctilde[0, 0] = 0
    return Ctilde


def convolve_with_C(g: np.ndarray, Ctilde: np.ndarray, res: int) -> np.ndarray:
    """Convolve patterns g (..., res, res) with the kernel given by its FT."""
    gtilde = np.fft.fft2(g, [res, res])
    gconv = np.real(np.fft.ifft2(gtilde * Ctilde))
    return np.roll(np.roll(gconv, res // 2 + 1, axis=-1), res // 2 + 1, axis=-2)


def run_pattern_formation(Ctilde: np.ndarray, res: int, nonlinearity: str = "relu",
                          n_units: int = 32, n_steps: int = 1000,
                          lr: float | None = None, seed: int = 0,
                          snapshot_every: int | None = None):
    """Single-cell pattern-formation dynamics (pattern_formation.ipynb).

    G <- G + lr * (-G + conv(G, Ctilde)); G <- phi(G)

    Args:
        nonlinearity: "tanh" (symmetry-preserving, stripes) or "relu"
            (symmetry-breaking, blobs). lr defaults to the notebook values
            (1e-4 for tanh, 5e-3 for relu).
        snapshot_every: If given, also return snapshots at this interval.

    Returns:
        G (n_units, res, res), and optionally snapshots (n_snaps, n_units, res, res).
    """
    if lr is None:
        lr = 1e-4 if nonlinearity == "tanh" else 5e-3
    phi = np.tanh if nonlinearity == "tanh" else (lambda x: np.maximum(x, 0))

    rng = np.random.RandomState(seed)
    G = rng.randn(n_units, res, res) * 1e-8
    snapshots = []
    for i in range(n_steps):
        G += lr * (-G + convolve_with_C(G, Ctilde, res))
        G = phi(G)
        if snapshot_every and (i + 1) % snapshot_every == 0:
            snapshots.append(G.copy())
    if snapshot_every:
        return G, np.asarray(snapshots)
    return G

In [ ]:
# --------------------------------------- ratemaps and grid scores, all units
t0 = time.time()
ratemaps, occupancy = compute_ratemaps(model, dataset, res=20, n_batches=50)
scores = score_ratemaps(ratemaps, box_width=2.2)
print(f"{len(ratemaps)} units scored in {time.time()-t0:.0f}s; "
      f"occupancy per bin: {occupancy.mean():.0f} samples")

order = np.argsort(-scores["score_60"])          # units sorted by grid score
n_grid_cells = int((scores["score_60"] > 0.5).sum())
print(f"grid cells (score_60 > 0.5): {n_grid_cells} / {len(ratemaps)} "
      f"({100*n_grid_cells/len(ratemaps):.0f}%)")

In [ ]:
# ----------------- ratemap + autocorrelogram panel, best and median units
def smooth(rm, sigma=0.8):
    return scipy.ndimage.gaussian_filter(np.nan_to_num(rm), sigma)

show = list(order[:18])                          # 18 highest grid scores
fig, axes = plt.subplots(3, 12, figsize=(11.5, 3.4), constrained_layout=True)
for k, unit in enumerate(show):
    ax_rm = axes[k // 6, 2 * (k % 6)]
    ax_sac = axes[k // 6, 2 * (k % 6) + 1]
    ax_rm.imshow(smooth(ratemaps[unit]), cmap="jet", origin="lower")
    ax_rm.set_title(f"unit {unit}\nscore {scores['score_60'][unit]:.2f}", fontsize=6)
    sac = scores["sacs"][unit]
    ax_sac.imshow(sac, cmap="jet", origin="lower")
    for ax in (ax_rm, ax_sac):
        ax.set_xticks([]); ax.set_yticks([])

fig.supxlabel("each pair: ratemap over the 2.2 m box  |  its spatial autocorrelogram", fontsize=8)
fig.suptitle("Hidden units of the trained RNN, ranked by grid score", fontsize=10)
fig.savefig(FIG_DIR / "ratemaps.png", bbox_inches="tight")
plt.show()

In [ ]:
# --------------------------- score distributions and the hexagon/square plane
fig, axes = plt.subplots(1, 2, figsize=(7.5, 2.8), width_ratios=[1, 1.2])

ax = axes[0]
ax.hist(scores["score_60"], bins=60, color="tab:blue", alpha=0.8)
ax.axvline(0.5, color="k", ls="--", lw=0.8, label="grid-cell threshold 0.5")
ax.set(xlabel="grid score (60° symmetry)", ylabel="number of units",
       title=f"grid-score distribution — {100*np.mean(scores['score_60']>0.5):.0f}% grid cells")
ax.legend(fontsize=7)

ax = axes[1]
ax.scatter(scores["score_90"], scores["score_60"], s=2, alpha=0.3, color="tab:blue")
ax.axhline(0.5, color="0.6", ls="--", lw=0.7)
ax.axvline(0.5, color="0.6", ls="--", lw=0.7)
ax.set(xlabel="squareness (90° symmetry score)", ylabel="hexagonality (60° symmetry score)",
       title="hexagonal, not square, lattices")
ax.annotate("hexagonal grids", xy=(0.0, 1.2), fontsize=7, color="tab:blue")

fig.savefig(FIG_DIR / "grid_scores.png", bbox_inches="tight")
plt.show()

**Reading the result.** A large fraction of the 4096 hidden units develop clean,
regular hexagonal firing fields — different units tile the arena with different phases,
and the autocorrelograms show the tell-tale six satellites. In the hexagonality–squareness
plane the population hugs the 60° axis: these are hex grids, not the square lattices of
Cueva & Wei (2018) or the irregular fields of Banino et al. (2018). This reproduces the
paper's Fig. 1C–E qualitatively.

Finally, a sanity check that the network actually *path integrates* rather than memorizing
maps: decode position from its readout on held-out test trajectories of the training
length (0.4 s). Because the network only ever sees velocity inputs after being seeded with
the initial place-cell code, accurate decoding means the recurrent dynamics carried the
position forward on their own.

In [ ]:
# --------------------------- decode position on held-out test trajectories
test_batch = dataset.sample_batch()              # fresh batch, training length (20 steps)
model.eval()
with torch.no_grad():
    out = model(test_batch["inputs"].to(DEVICE),
                init_pc=test_batch["init_pc"].to(DEVICE))
    pred_pos = place_cells.decode(out.outputs).cpu().numpy()      # (B, T, 2)
true_pos = test_batch["positions"].numpy()                        # (B, T, 2)
decode_err = np.sqrt(((pred_pos - true_pos) ** 2).sum(-1))        # (B, T), meters
print(f"test batch: {true_pos.shape[0]} trajectories x {true_pos.shape[1]} steps, "
      f"mean error {100*decode_err.mean():.1f} cm, median {100*np.median(decode_err):.1f} cm")

fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.2), width_ratios=[1.4, 1])
ax = axes[0]
for i in range(4):                               # four example trajectories
    ax.plot(true_pos[i, :, 0], true_pos[i, :, 1], lw=1.2, color="0.3",
            label="true path" if i == 0 else None)
    ax.plot(pred_pos[i, :, 0], pred_pos[i, :, 1], ".-", ms=3, lw=0.9, color="tab:orange",
            alpha=0.85, label="decoded from network" if i == 0 else None)
    ax.scatter(*true_pos[i, 0], c="tab:green", s=16, zorder=5)
ax.set(xlim=(-1.1, 1.1), ylim=(-1.1, 1.1), xlabel="x (m)", ylabel="y (m)",
       title="path integration on held-out trajectories", aspect="equal")
ax.legend(fontsize=7, loc="upper right")

ax = axes[1]
ax.plot(np.arange(decode_err.shape[1]) * 0.02, 100 * decode_err.T, lw=0.3,
        color="tab:red", alpha=0.15)
ax.plot(np.arange(decode_err.shape[1]) * 0.02, 100 * decode_err.mean(0), lw=1.2,
        color="tab:red", label="mean over batch")
ax.set(xlabel="time within trajectory (s)", ylabel="decoding error (cm)",
       title=f"mean error {100*decode_err.mean():.1f} cm (box width 220 cm)")
ax.legend(fontsize=7)
fig.savefig(FIG_DIR / "decoding.png", bbox_inches="tight")
plt.show()

## Why hexagons? A pattern-formation theory

The trained network works, but *why a hexagon*? The paper's answer (its Fig. 3) recasts
learning as a pattern-formation problem, and it is reproducible here in a few lines of
NumPy — no training required.

**Step 1 — the position-encoding objective.** Strip the trained models to their shared
core: choose hidden spatial maps $g(x)$ that can generate the place-cell code $p(x)$
through one linear synaptic layer, with minimal total activity. With
$\Sigma = P P^{\top}$ the spatial correlation matrix of the place code (entries
$\Sigma_{xx'} = \sum_i p_i(x) p_i(x')$), the optimal maps are the top eigenvectors of
$\Sigma$, subject to an activity penalty $\sigma(g)$ that prefers negative values to be
rare.

**Step 2 — the eigenvectors of $\Sigma$ are plane waves on a ring.** Because $\Sigma$
depends (approximately) only on the displacement $x - x'$, its eigenvectors are Fourier
modes, and its eigenvalues are the Fourier transform of the tuning profile — which for a
DoG kernel is *band-pass*: power concentrates on an annulus of wavevectors of one
characteristic wavelength, set by the tuning width $\sigma$. Learning therefore amplifies
every direction of plane wave of that wavelength equally. Wide tuning gives a small ring
that only the few cardinal-axis modes reach — square grids (Cueva & Wei). A narrow DoG
ring admits many modes — the irregular fields of Banino et al.

**Step 3 — nonnegativity selects the triangle.** Any combination of ring modes is equally
accurate; the tie is broken by $\sigma(g)$. A nonnegativity-favoring penalty rewards
superpositions whose peaks reinforce and whose negative lobes cancel — and three
wavevectors $k + k' + k'' = 0$ forming an equilateral triangle do exactly this, because the
cubic (three-wave) interaction only survives when the nonlinearity breaks the
$g \to -g$ symmetry. ReLU breaks it; tanh does not. Hence: tanh → stripe patterns,
ReLU → hexagonal lattices.

The simulations below integrate the resulting dynamics directly,

$$
\dot{G} = -G + \mathrm{conv}(G, \tilde{\Sigma}), \qquad G \leftarrow \phi(G),
$$

from tiny random initial conditions: each hidden map grows by convolution with the
place-cell covariance kernel $\tilde\Sigma$ and is then passed through the nonlinearity
$\phi$. This is the exact numerical experiment of the paper's `pattern_formation.ipynb`.

In [ ]:
# ------------------- the place-cell covariance kernel and its Fourier annulus
res_cov = 55
C = place_cells.covariance(res=res_cov)          # Sigma(dx, dy), circulant-averaged
Ctilde = covariance_ft(C)                        # DC mode removed, as in the paper

fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.0))
ax = axes[0]
im = ax.imshow(C, cmap="jet", interpolation="gaussian")
ax.set_title(r"$\Sigma$: place-code spatial covariance", fontsize=9)
ax.set_xticks([]); ax.set_yticks([])

ax = axes[1]
width = 6
idxs = np.arange(-width + 1, width)
fx, fy = np.meshgrid(np.arange(2 * width - 1), np.arange(2 * width - 1))
ring_power = np.abs(Ctilde)[np.ix_(np.mod(idxs, res_cov), np.mod(idxs, res_cov))]
ax.scatter(fx - width + 1, fy - width + 1, c=ring_power.ravel(),
           s=260, cmap="Oranges", marker="s", edgecolor="0.5", linewidth=0.3)
ax.set_aspect("equal"); ax.axis("off")
ax.set_title(r"$\tilde{\Sigma}$: power lives on a ring", fontsize=9)

fig.savefig(FIG_DIR / "kernel.png", bbox_inches="tight")
plt.show()

In [ ]:
# ------------- pattern formation: tanh -> stripes, ReLU -> hexagonal blobs
t0 = time.time()
G_tanh, snaps_tanh = run_pattern_formation(Ctilde, res=res_cov, nonlinearity="tanh",
                                           n_units=32, n_steps=1000, lr=1e-4,
                                           seed=1, snapshot_every=250)
G_relu, snaps_relu = run_pattern_formation(Ctilde, res=res_cov, nonlinearity="relu",
                                           n_units=32, n_steps=1000, lr=5e-3,
                                           seed=1, snapshot_every=250)
print(f"simulations done in {time.time()-t0:.0f}s")

fig, axes = plt.subplots(4, 8, figsize=(9.5, 4.6))
for k in range(8):
    axes[0, k].imshow(G_tanh[k], cmap="jet"); axes[0, k].set_title(f"unit {k}", fontsize=6)
    axes[1, k].imshow(G_tanh[8 + k], cmap="jet")
    axes[2, k].imshow(G_relu[k], cmap="jet")
    axes[3, k].imshow(G_relu[8 + k], cmap="jet")
for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([])
axes[0, 0].set_ylabel("tanh\n(symmetry-\npreserving)", fontsize=7)
axes[2, 0].set_ylabel("ReLU\n(symmetry-\nbreaking)", fontsize=7)

fig.suptitle("Pattern formation with the covariance kernel: "
             "stripes under tanh, hexagonal blobs under ReLU", fontsize=10)
fig.savefig(FIG_DIR / "pattern_formation.png", bbox_inches="tight")
plt.show()

**Reading the result.** Both runs start from the same microscopic noise and are driven
by the same covariance kernel. Under tanh the growth is symmetric in
$g \to -g$, so each unit converges to a single plane wave — stripes. Under ReLU the
symmetry is broken, three wavevectors at 120° lock together, and the units crystallize into
hexagonal lattices of blobs (deformed near the boundary, as the paper notes). The same two
ingredients — the DoG kernel setting the ring, nonnegativity selecting the triangle —
explain the trained RNN's grids above.

## Summary

* A plain ReLU RNN trained to path integrate, with a center-surround place-cell readout,
  spontaneously develops hexagonal grid cells — reproducing the central finding of
  Sorscher et al. (2023) inside the NeuralRNN framework (a `CTRNN` with $\alpha=1$, a
  framework `Trainer`, and a small custom `Objective` for the soft-target cross-entropy).
* The learned representation is not a lookup table: position can be decoded from the
  readout along held-out trajectories, and grid scores quantify the six-fold symmetry.
* The pattern-formation theory is reproduced in silico: the place-code covariance kernel
  is band-pass, and the nonlinearity alone decides between stripes (tanh) and hexagons
  (ReLU).

*Differences from the reference implementation*: none in the model, data, loss, or
hyperparameters. The training budget matches the paper's default (100k steps). The
reference also trains LSTM variants and analyses of attractor topology (its Fig. 4) that
are outside this notebook's scope.